# MRMS Hawaii Data Download

This notebook downloads MRMS (Multi-Radar/Multi-Sensor) 1-hour QPE data for Hawaii.

**Key Updates:**
- Files are gzip compressed (`.grib2.gz`)
- Product path includes `_00.00` suffix
- Automatic decompression and conversion to NetCDF

In [1]:
from datetime import datetime
from download_mrms_hawaii import MRMSHawaiiDownloader

In [2]:
# Test: List available products in Hawaii bucket
output_dir = r"C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\data\MRMS"
downloader = MRMSHawaiiDownloader(output_dir=output_dir, verbose=True)

# List products
downloader.list_available_products()

# List recent dates for 1-hour QPE
# NOTE: Pass 'MultiSensor_QPE_01H_Pass2' WITHOUT the _00.00 suffix
# The code adds _00.00 suffix automatically
downloader.list_available_dates(product='MultiSensor_QPE_01H_Pass2', limit=20)


Available MRMS products in HAWAII:
  - BREF_1HR_MAX_00.50
  - BrightBandBottomHeight_00.00
  - BrightBandTopHeight_00.00
  - CREF_1HR_MAX_00.50
  - EchoTop_18_00.50
  - EchoTop_30_00.50
  - EchoTop_50_00.50
  - EchoTop_60_00.50
  - FLASH_CREST_MAXSOILSAT_00.00
  - FLASH_CREST_MAXSTREAMFLOW_00.00
  - FLASH_CREST_MAXUNITSTREAMFLOW_00.00
  - FLASH_HP_MAXSTREAMFLOW_00.00
  - FLASH_HP_MAXUNITSTREAMFLOW_00.00
  - FLASH_QPE_ARI01H_00.00
  - FLASH_QPE_ARI03H_00.00
  - FLASH_QPE_ARI06H_00.00
  - FLASH_QPE_ARI12H_00.00
  - FLASH_QPE_ARI24H_00.00
  - FLASH_QPE_ARI30M_00.00
  - FLASH_QPE_ARIMAX_00.00
  - FLASH_QPE_FFG01H_00.00
  - FLASH_QPE_FFG03H_00.00
  - FLASH_QPE_FFG06H_00.00
  - FLASH_QPE_FFGMAX_00.00
  - FLASH_SAC_MAXSOILSAT_00.00
  - FLASH_SAC_MAXSTREAMFLOW_00.00
  - FLASH_SAC_MAXUNITSTREAMFLOW_00.00
  - GaugeInflIndex_01H_Pass1_00.00
  - GaugeInflIndex_01H_Pass2_00.00
  - GaugeInflIndex_03H_Pass1_00.00
  - GaugeInflIndex_03H_Pass2_00.00
  - GaugeInflIndex_06H_Pass1_00.00
  - GaugeInflInde

['20201014',
 '20201015',
 '20201016',
 '20201017',
 '20201018',
 '20201019',
 '20201020',
 '20201021',
 '20201022',
 '20201023',
 '20201024',
 '20201025',
 '20201026',
 '20201027',
 '20201028',
 '20201029',
 '20201030',
 '20201031',
 '20201101',
 '20201102',
 '20201103',
 '20201104',
 '20201105',
 '20201106',
 '20201107',
 '20201108',
 '20201109',
 '20201110',
 '20201111',
 '20201112',
 '20201113',
 '20201114',
 '20201115',
 '20201116',
 '20201117',
 '20201118',
 '20201119',
 '20201120',
 '20201121',
 '20201122',
 '20201123',
 '20201124',
 '20201125',
 '20201126',
 '20201127',
 '20201128',
 '20201129',
 '20201130',
 '20201201',
 '20201202',
 '20201203',
 '20201204',
 '20201205',
 '20201206',
 '20201207',
 '20201208',
 '20201209',
 '20201210',
 '20201211',
 '20201212',
 '20201213',
 '20201214',
 '20201215',
 '20201216',
 '20201217',
 '20201218',
 '20201219',
 '20201220',
 '20201221',
 '20201222',
 '20201223',
 '20201224',
 '20201225',
 '20201226',
 '20201227',
 '20201228',
 '20201229',

### Debug: Check Specific File Availability

This cell tests whether a specific file can be found and shows the exact S3 path being checked.

In [3]:
# Test: Check availability of a specific file to debug
from datetime import datetime

test_dt = datetime(2026, 3, 10, 0, 0)  # March 10, 2026 at 00:00 UTC
print(f"Testing file availability for: {test_dt}")

# Check what path it's looking for
product = 'MultiSensor_QPE_01H_Pass2'
expected_path = downloader.get_aws_path(test_dt, product)
print(f"\nExpected S3 path: s3://{expected_path}")

# Check if it exists
availability = downloader.check_file_availability(test_dt, product)
print(f"\nAvailability result: {availability}")

if availability['available']:
    print(f"✓ File found!")
    print(f"  Size: {availability['size_mb']:.2f} MB")
else:
    print(f"❌ File not found")
    print("\nTrying to list what's actually in the directory...")
    # List what's actually in that date directory
    import s3fs
    fs = s3fs.S3FileSystem(anon=True)
    date_str = test_dt.strftime('%Y%m%d')
    dir_path = f"noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/{date_str}"
    try:
        files = fs.ls(dir_path)
        print(f"\nFiles in {dir_path}:")
        for f in files[:5]:  # Show first 5
            print(f"  - {f.split('/')[-1]}")
        if len(files) > 5:
            print(f"  ... and {len(files) - 5} more files")
    except Exception as e:
        print(f"Error listing directory: {e}")

Testing file availability for: 2026-03-10 00:00:00

Expected S3 path: s3://noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260310/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260310-000000.grib2.gz
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260310/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260310-000000.grib2.gz

Availability result: {'source': 'aws', 'path': 'noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260310/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260310-000000.grib2.gz', 'size_mb': 0.05786323547363281, 'available': True}
✓ File found!
  Size: 0.06 MB


## Test Connection and List Available Data

First, let's verify we can connect to the Hawaii MRMS bucket and see what's available.

In [4]:
def download_mrms_hawaii_qpe(start_date, end_date, output_dir=None):
    """
    Simple function to download MRMS 1-hour QPE for Hawaii.
    
    Args:
        start_date (str or datetime): Start date (YYYY-MM-DD or datetime object)
        end_date (str or datetime): End date (YYYY-MM-DD or datetime object)
        output_dir (str, optional): Output directory path
        
    Returns:
        list: Paths to downloaded files
        
    Example:
        >>> # Download for March 2, 2026
        >>> files = download_mrms_hawaii_qpe("2026-03-02", "2026-03-02")
        >>> print(f"Downloaded {len(files)} files")
        
        >>> # Download for a week
        >>> files = download_mrms_hawaii_qpe("2026-03-01", "2026-03-07")
    """
    # Parse dates if strings
    if isinstance(start_date, str):
        start_time = datetime.strptime(start_date, '%Y-%m-%d')
    else:
        start_time = start_date
    
    if isinstance(end_date, str):
        end_time = datetime.strptime(end_date, '%Y-%m-%d')
        end_time = end_time.replace(hour=23, minute=59)
    else:
        end_time = end_date
    
    # Initialize downloader
    downloader = MRMSHawaiiDownloader(output_dir=output_dir, verbose=True)
    
    # Download data
    files = downloader.download_time_range(
        start_time=start_time,
        end_time=end_time,
        interval_minutes=60,  # Hourly
        product='MultiSensor_QPE_01H_Pass2',  # Quality-controlled
        subset_hawaii=True  # Only Hawaii domain
    )
    
    return files

## Download MRMS Data

The function below downloads MRMS 1-hour QPE files for a specified date range.

**Note:** Files are automatically decompressed from `.grib2.gz` format and converted to NetCDF.

In [5]:
# Download MRMS data for Hawaii for a specific date range
output_dir = r"C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\data\MRMS"

files = download_mrms_hawaii_qpe(
    start_date="2026-05-18",
    end_date="2026-05-19",
    output_dir=output_dir
)

print(f"\nDownloaded {len(files)} files")

if files:
    print(f"First file: {files[0]}")


Time range: 2026-05-18 00:00:00 to 2026-05-19 23:59:00
Interval: 60 minutes
Output directory: C:\Users\domenicbrooks\OneDrive - Research Triangle Institute\CIROH Flash Flooding\data\MRMS

Total files to download: 48



  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260518/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260518-000000.grib2.gz
  ⬇ Downloading from AWS S3 (0.1 MB)...
  📦 Decompressing...


  ✓ Saved: mrms_qpe_1hr_hawaii_20260518_0000.nc (5.4 MB)
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260518/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260518-010000.grib2.gz
  ⬇ Downloading from AWS S3 (0.1 MB)...
  📦 Decompressing...


  ✓ Saved: mrms_qpe_1hr_hawaii_20260518_0100.nc (5.4 MB)
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260518/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260518-020000.grib2.gz
  ⬇ Downloading from AWS S3 (0.1 MB)...


  📦 Decompressing...
  ✓ Saved: mrms_qpe_1hr_hawaii_20260518_0200.nc (5.4 MB)
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260518/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260518-030000.grib2.gz
  ⬇ Downloading from AWS S3 (0.1 MB)...
  📦 Decompressing...


  ✓ Saved: mrms_qpe_1hr_hawaii_20260518_0300.nc (5.4 MB)
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260518/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260518-040000.grib2.gz
  ⬇ Downloading from AWS S3 (0.1 MB)...
  📦 Decompressing...


... (41 outputs left out)

  ✓ Saved: mrms_qpe_1hr_hawaii_20260519_2100.nc (5.4 MB)
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260519/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260519-220000.grib2.gz
  ⬇ Downloading from AWS S3 (0.0 MB)...
  📦 Decompressing...


  ✓ Saved: mrms_qpe_1hr_hawaii_20260519_2200.nc (5.4 MB)
  Found: noaa-mrms-pds/HAWAII/MultiSensor_QPE_01H_Pass2_00.00/20260519/MRMS_MultiSensor_QPE_01H_Pass2_00.00_20260519-230000.grib2.gz
  ⬇ Downloading from AWS S3 (0.0 MB)...
  📦 Decompressing...


  ✓ Saved: mrms_qpe_1hr_hawaii_20260519_2300.nc (5.4 MB)

DOWNLOAD SUMMARY

Successfully downloaded: 48 files
Failed: 0 files

  Removed: temp_20260518_0000.grib2.5b7b6.idx
  Removed: temp_20260518_0100.grib2.5b7b6.idx
  Removed: temp_20260518_0200.grib2.5b7b6.idx
  Removed: temp_20260518_0300.grib2.5b7b6.idx
  Removed: temp_20260518_0400.grib2.5b7b6.idx
  Removed: temp_20260518_0500.grib2.5b7b6.idx
  Removed: temp_20260518_0600.grib2.5b7b6.idx
  Removed: temp_20260518_0700.grib2.5b7b6.idx
  Removed: temp_20260518_0800.grib2.5b7b6.idx
  Removed: temp_20260518_0900.grib2.5b7b6.idx
  Removed: temp_20260518_1000.grib2.5b7b6.idx
  Removed: temp_20260518_1100.grib2.5b7b6.idx
  Removed: temp_20260518_1200.grib2.5b7b6.idx
  Removed: temp_20260518_1300.grib2.5b7b6.idx
  Removed: temp_20260518_1400.grib2.5b7b6.idx
  Removed: temp_20260518_1500.grib2.5b7b6.idx
  Removed: temp_20260518_1600.grib2.5b7b6.idx
  Removed: temp_20260518_1700.grib2.5b7b6.idx
  Removed: temp_20260518_1800.grib2.5b7b6.idx

  Removed: _tmp_20260225_2300.grib2.5b7b6.idx
  Removed: _tmp_20260226_0000.grib2.5b7b6.idx
  Removed: _tmp_20260226_0100.grib2.5b7b6.idx
  Removed: _tmp_20260226_0200.grib2.5b7b6.idx
  Removed: _tmp_20260226_0300.grib2.5b7b6.idx
  Removed: _tmp_20260226_0400.grib2.5b7b6.idx
  Removed: _tmp_20260226_0500.grib2.5b7b6.idx
  Removed: _tmp_20260226_0600.grib2.5b7b6.idx
  Removed: _tmp_20260226_0700.grib2.5b7b6.idx
  Removed: _tmp_20260226_0800.grib2.5b7b6.idx
  Removed: _tmp_20260226_0900.grib2.5b7b6.idx
  Removed: _tmp_20260226_1000.grib2.5b7b6.idx
  Removed: _tmp_20260226_1100.grib2.5b7b6.idx
  Removed: _tmp_20260226_1200.grib2.5b7b6.idx
  Removed: _tmp_20260226_1300.grib2.5b7b6.idx
  Removed: _tmp_20260226_1400.grib2.5b7b6.idx
  Removed: _tmp_20260226_1500.grib2.5b7b6.idx
  Removed: _tmp_20260226_1600.grib2.5b7b6.idx
  Removed: _tmp_20260226_1700.grib2.5b7b6.idx
  Removed: _tmp_20260226_1800.grib2.5b7b6.idx
  Removed: _tmp_20260226_1900.grib2.5b7b6.idx
  Removed: _tmp_20260226_2000.grib